# Evaluating Model Performance

We use separate data for learning, tuning decisions, and final evaluation.

![Training, validation, and test data split](diagrams/07-data-splits.svg)

- **Training set:** updates model parameters
- **Validation set:** helps choose learning rate, architecture, and other hyperparameters
- **Test set:** checks final performance on untouched data

## What Does Validation Actually Do?

Validation does **not** teach the model directly. There is no `loss.backward()` or `optimizer.step()` in the validation loop, so weights and biases stay unchanged.

It asks: **After learning from the training data, does the model also work on different data?**

```text
Training set   -> homework used to learn
Validation set -> practice exam used to adjust the study plan
Test set       -> final exam used to report the result
```

Validation and testing use almost the same calculation, but their results are used differently:

| Dataset | Updates weights? | Used repeatedly? | Purpose |
|---|---:|---:|---|
| Training | Yes | Yes | Learn weights and biases |
| Validation | No | Yes | Choose settings and models |
| Test | No | Ideally once | Report final performance |

### Where Does Hyperparameter Tuning Happen?

The validation loop only **measures** loss and accuracy. Tuning happens when we train fresh models with different settings and compare their validation results.

```python
# Try one configuration
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,       # learning rate, sometimes called alpha
    momentum=0.9, # momentum is the optimizer's inertia
)
```

The development process looks like this:

![Hyperparameter-tuning workflow](diagrams/07-hyperparameter-tuning.svg)

If we repeatedly check the test set and make choices from its results, it becomes another validation set. It is no longer an unbiased final test.

## Prepare Example Data

This synthetic dataset has seven animal features and three classes. The split is `60%` training, `20%` validation, and `20%` testing.

In [9]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchmetrics.classification import MulticlassAccuracy

torch.manual_seed(42)

features = torch.randn(150, 7)  # 150 animals, 7 features each

# Create learnable class labels for this example
teacher_weights = torch.tensor([
    [ 1.0, -0.5,  0.2],
    [ 0.4,  0.8, -0.6],
    [-0.7,  0.1,  0.9],
    [ 0.3, -0.4,  0.7],
    [ 0.8,  0.2, -0.5],
    [-0.2,  0.9,  0.1],
    [ 0.5, -0.8,  0.4],
])

In [10]:

targets_tensor = (features @ teacher_weights) 
targets = targets_tensor.argmax(dim=-1)
print(targets_tensor)
print(len(targets))
print(targets)

tensor([[ 2.0278e+00,  2.1770e-01, -1.6501e+00],
        [-4.4344e+00,  3.4219e-02,  1.7447e+00],
        [-2.5263e+00,  2.1251e+00,  7.4454e-01],
        [ 2.1884e+00, -5.0119e-01,  2.0899e+00],
        [-8.2603e-01, -1.1022e+00,  8.1997e-01],
        [ 2.4235e+00, -3.1216e+00,  6.1388e-01],
        [-9.3855e-01, -9.7254e-01,  7.2695e-01],
        [-3.9349e-01,  7.3373e-01, -1.0183e+00],
        [-2.1812e+00,  3.0420e+00, -1.2690e-01],
        [ 2.2781e+00,  2.3722e-01,  1.4272e+00],
        [-1.8180e-02,  1.2579e+00, -5.2171e-01],
        [-1.2910e+00, -2.3297e+00,  3.2853e-01],
        [ 1.2943e-01,  8.0000e-01,  1.5775e-02],
        [ 2.0276e+00, -4.8487e+00,  1.0459e+00],
        [ 1.3734e+00, -5.8214e-01,  4.7210e-01],
        [ 1.7362e+00,  3.5565e-01, -5.0440e-01],
        [-5.6726e-01,  9.3443e-01, -9.4490e-01],
        [ 9.5635e-01,  5.6341e-01,  2.8606e-01],
        [-1.1549e+00, -1.4378e+00,  2.8943e+00],
        [ 2.6753e+00, -3.0548e+00,  1.4320e+00],
        [ 5.4936e-01

In [ ]:
dataset = TensorDataset(features, targets)
train_dataset, validation_dataset, test_dataset = random_split(
    dataset,
    lengths=[90, 30, 30],
    generator=torch.Generator().manual_seed(42),
)

trainloader = DataLoader(train_dataset, batch_size=15, shuffle=True)  # For you to have a train datasets it has to have X and y
validationloader = DataLoader(validation_dataset, batch_size=15)
testloader = DataLoader(test_dataset, batch_size=15)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 90
Validation samples: 30
Test samples: 30


## Model and Metrics

We track two values:

- **Loss:** how wrong the class scores are
- **Accuracy:** percentage of correct class predictions

In [12]:
model = nn.Sequential(
    nn.Linear(7, 16),
    nn.ReLU(),
    nn.Linear(16, 3),  # 3 raw class scores per animal
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# average="micro" calculates total correct predictions / total samples
training_accuracy = MulticlassAccuracy(num_classes=3, average="micro")
validation_accuracy = MulticlassAccuracy(num_classes=3, average="micro")

## Training and Validation Loop

One epoch contains two different loops:

![Training and validation loops](diagrams/07-training-validation-loop.svg)

In [ ]:
number_of_epochs = 15
training_loss_history = []
validation_loss_history = []

for epoch in range(number_of_epochs):
    # TRAINING ---------------------------------------------------------
    model.train()  # enable training behavior for layers such as Dropout
    training_loss = 0.0
    training_accuracy.reset()  # clear accuracy counts from the previous epoch

    for batch_inputs, batch_labels in trainloader:
        optimizer.zero_grad()  # clear gradients from the previous batch

        scores = model(batch_inputs)              # forward pass
        loss = criterion(scores, batch_labels)    # compare scores with targets
        loss.backward()                           # calculate parameter gradients
        optimizer.step()                          # update weights and biases

        training_loss += loss.item()  # tensor scalar -> Python number

        # dim=-1 means choose from the 3 scores in the last dimension
        predicted_classes = scores.argmax(dim=-1)
        training_accuracy.update(predicted_classes, batch_labels)

    mean_training_loss = training_loss / len(trainloader)
    epoch_training_accuracy = training_accuracy.compute().item()
    training_loss_history.append(mean_training_loss)

    # VALIDATION -------------------------------------------------------
    model.eval()  # use evaluation behavior for layers such as Dropout
    validation_loss = 0.0
    validation_accuracy.reset()  # clear counts from the previous epoch

    # Validation does not update parameters, so gradients are unnecessary
    with torch.no_grad():
        for batch_inputs, batch_labels in validationloader:
            scores = model(batch_inputs)
            loss = criterion(scores, batch_labels)
            validation_loss += loss.item()

            predicted_classes = scores.argmax(dim=-1)
            validation_accuracy.update(predicted_classes, batch_labels)

    mean_validation_loss = validation_loss / len(validationloader)
    epoch_validation_accuracy = validation_accuracy.compute().item()
    validation_loss_history.append(mean_validation_loss)

    print(
        f"Epoch {epoch + 1:>2}/{number_of_epochs} | "
        f"train loss: {mean_training_loss:.4f}, "
        f"train accuracy: {epoch_training_accuracy:.1%} | "
        f"validation loss: {mean_validation_loss:.4f}, "
        f"validation accuracy: {epoch_validation_accuracy:.1%}"
    )

# Prepare the model in case another training epoch follows
model.train()

Epoch  1/15 | train loss: 1.1139, train accuracy: 36.7% | validation loss: 1.0229, validation accuracy: 46.7%
Epoch  2/15 | train loss: 1.0338, train accuracy: 48.9% | validation loss: 0.9852, validation accuracy: 63.3%
Epoch  3/15 | train loss: 0.9670, train accuracy: 53.3% | validation loss: 0.9389, validation accuracy: 66.7%
Epoch  4/15 | train loss: 0.9053, train accuracy: 65.6% | validation loss: 0.8803, validation accuracy: 70.0%
Epoch  5/15 | train loss: 0.8355, train accuracy: 77.8% | validation loss: 0.8248, validation accuracy: 70.0%
Epoch  6/15 | train loss: 0.7589, train accuracy: 76.7% | validation loss: 0.7619, validation accuracy: 73.3%
Epoch  7/15 | train loss: 0.6819, train accuracy: 84.4% | validation loss: 0.6973, validation accuracy: 83.3%
Epoch  8/15 | train loss: 0.6038, train accuracy: 88.9% | validation loss: 0.6404, validation accuracy: 86.7%
Epoch  9/15 | train loss: 0.5366, train accuracy: 88.9% | validation loss: 0.5831, validation accuracy: 83.3%
Epoch 10/1

Sequential(
  (0): Linear(in_features=7, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=3, bias=True)
)

### Why Both `eval()` and `no_grad()`?

They do different jobs:

- `model.eval()` changes the behavior of layers such as Dropout and BatchNorm.
- `torch.no_grad()` stops gradient tracking to save memory and computation.
- `model.train()` switches those layers back to training behavior.

## Overfitting

Overfitting means the model memorizes training patterns that do not generalize.

```text
Training loss:   keeps decreasing
Validation loss: decreases, then starts increasing  <- warning sign
```

Use validation performance to choose settings or when to stop training. Keep the test set untouched until the final model is chosen.

## Final Test Evaluation

The test loop looks like validation, but it is run only after model choices are finished.

In [14]:
test_accuracy = MulticlassAccuracy(num_classes=3, average="micro")
test_loss = 0.0

model.eval()  # switch to evaluation behavior
with torch.no_grad():  # no parameter updates during testing
    for batch_inputs, batch_labels in testloader:
        scores = model(batch_inputs)
        loss = criterion(scores, batch_labels)
        test_loss += loss.item()

        predicted_classes = scores.argmax(dim=-1)
        test_accuracy.update(predicted_classes, batch_labels)

mean_test_loss = test_loss / len(testloader)
final_test_accuracy = test_accuracy.compute().item()
test_accuracy.reset()  # clear stored predictions and targets

print(f"Test loss: {mean_test_loss:.4f}")
print(f"Test accuracy: {final_test_accuracy:.1%}")

Test loss: 0.3972
Test accuracy: 86.7%


## Quick Look

```python
# Training
model.train()
optimizer.zero_grad()
scores = model(inputs)
loss = criterion(scores, labels)
loss.backward()
optimizer.step()

# Validation or testing
model.eval()
with torch.no_grad():
    scores = model(inputs)
    loss = criterion(scores, labels)

predicted_classes = scores.argmax(dim=-1)
accuracy.update(predicted_classes, labels)
epoch_accuracy = accuracy.compute()
accuracy.reset()
```